#### Importing required libraries 

In [20]:
import pandas as pd
from sklearn.metrics import recall_score, precision_score,f1_score precision_recall_fscore_support
from tqdm import tqdm
from utils import Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning
import numpy as np
import warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
warnings.filterwarnings('ignore')

#### Testing a single load 

In [3]:
train_dataset = 'charlie_hebdo'
test_dataset = 'ottawashooting'
time_cut =3*60*24
processor = Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning(train_dataset,\
           test_dataset, time_cut=time_cut,test_size=0.7)

processor.load_data()
processor.process_data()
train,test = processor.get_final_dataframes()

rumour
1    307
0    293
Name: count, dtype: int64


In [15]:
train.head()

,followers,favorite_count,retweet_count,first_time_diff,replies,no_verified,verified,embeddings_avg,rumour,min_since_fst_post
0,-0.088385,-0.508065,-0.162281,3.614367,-0.444444,1,0,"[-0.12270056130364537, 0.01583862374536693, -0...",1,110710.30
1,0.034307,-0.314516,1.271930,1.043478,-0.444444,0,1,"[-0.12335950043052435, -0.055849663292368255, ...",1,110712.02
2,0.418183,-0.500000,-0.302632,0.778828,-0.444444,0,1,"[-0.1364929385483265, -0.07159566258390744, -0...",1,110712.32
3,0.389279,-0.500000,-0.399123,-0.147448,-0.666667,0,1,"[-0.045377860377941816, -0.20127306692302227, ...",1,110714.00
4,1.230894,-0.362903,0.947368,-0.340265,0.111111,0,1,"[-0.03706469060853124, -0.1309182441327721, -0...",1,110715.43


In [5]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])


X_test  = test.drop(columns=['rumour'])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])

#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']
y_test =test['rumour']

In [9]:




# Model definition
class RumorDetectionLSTM(nn.Module):


    """
    Hybrid neural network model for rumor detection using embeddings and structured features.

    This model combines an LSTM network that processes precomputed text embeddings
    with fully connected layers that process additional handcrafted or metadata features.
    The outputs from these two branches are fused and passed through dense layers
    for binary rumor classification.

    Architecture Overview:
    ---------------------
    • Input features are divided into:
        - First 8 features: handcrafted / numerical indicators
        - Last 100 features: embedding vector representing textual content
    • A single-step LSTM processes the embedding to extract contextual semantics
    • Dense layers extract nonlinear structure from auxiliary features
    • Concatenated representation is classified with fully connected layers

    Parameters
    ----------
    embedding_dim : int, default=100
        Dimensionality of the input embedding vector.
    lstm_hidden_size : int, default=32
        Number of hidden units in the LSTM layer.
    dense_hidden_size : int, default=16
        Number of hidden units in the dense feature branch.

    Forward Input
    -------------
    x : torch.Tensor
        Tensor of shape (batch_size, 108) where:
        - x[:, :8] are handcrafted features
        - x[:, -100:] is a sentence/document embedding

    Returns
    -------
    torch.Tensor
        A 1D tensor of shape (batch_size,) containing probabilities in [0, 1]
        where values close to 1 indicate high likelihood of rumor.

    Notes
    -----
    - Assumes sequence length = 1 in embedding branch.
    - Uses sigmoid activation for binary classification tasks.
    """
    
    def __init__(self, embedding_dim=100, lstm_hidden_size=32, dense_hidden_size=16):
        super(RumorDetectionLSTM, self).__init__()
        
        # LSTM for the 100-dimensional embeddings
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=lstm_hidden_size, batch_first=True)
        
        # Dense layers for other features
        self.dense1 = nn.Linear(8, 16)  # 8 non-embedding features
        self.dense2 = nn.Linear(16, dense_hidden_size)
        
        # Combine LSTM and dense features
        self.fc1 = nn.Linear(lstm_hidden_size + dense_hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)
        
    def forward(self, x):
        # Separate embeddings and other features
        embeddings = x[:, -100:].unsqueeze(1)  # (batch, seq_len=1, embedding_dim)
        other_features = x[:, :8]  # First 8 features
        
        # LSTM output
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = lstm_out[:, -1, :]  # Get the last LSTM output
        
        # Dense layers for other features
        dense_out = torch.relu(self.dense1(other_features))
        dense_out = torch.relu(self.dense2(dense_out))
        
        # Concatenate LSTM and dense outputs
        combined = torch.cat((lstm_out, dense_out), dim=1)
        
        # Fully connected layers for classification
        x = torch.relu(self.fc1(combined))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze()


#### Example  training

In [14]:
# Assuming X_train, X_test, y_train, and y_test are available as numpy arrays
# Convert them to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Dataset and DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [21]:


# Model, criterion, optimizer initialization (as before)
model = RumorDetectionLSTM()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with loss and recall monitoring
epochs = 75  # Adjust as needed
train_recall_interval = 50  # Calculate train recall every 10 epochs
loss_interval = 50  # Print loss every 10 epochs

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Print loss every 10 epochs
    if (epoch + 1) % loss_interval == 0:
        model.eval()
        train_preds = []
        train_labels = []
        with torch.no_grad():
            for X_batch, y_batch in train_loader:
                output = model(X_batch)
                preds = (output >= 0.5).int()  # Binarize predictions
                train_preds.extend(preds.tolist())
                train_labels.extend(y_batch.tolist())
        
        train_recall = recall_score(train_labels, train_preds)
        train_precision = precision_score(train_labels, train_preds)
        
print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f},\
              Train Precision: {train_precision:.4f},Train Recall: {train_recall:.4f}")
    


# Final evaluation on test set with recall and precision
model.eval()
test_preds = []
test_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = (output >= 0.5).int()  # Binarize predictions
        test_preds.extend(preds.tolist())
        test_labels.extend(y_batch.tolist())

# Calculate final test recall and precision
test_recall = recall_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds)

print(f"Final Test Recall: {test_recall:.4f}")
print(f"Final Test Precision: {test_precision:.4f}")


Epoch 75, Train Loss: 68.8335,              Train Precision: 0.2774,Train Recall: 1.0000
Final Test Recall: 0.7948
Final Test Precision: 0.6759


#### Setting MLflow Experiment

In [ ]:
mlflow.set_experiment("LSTM  2025-11-04 Ottawa Shooting TF")

#### Loading dataset statistics to get the final time cut 

In [16]:
df_posts_by_time_cut = pd.read_csv('ottawa_shooting_posts_by_time_cut.csv')


In [17]:
time_cut_last_post = int(df_posts_by_time_cut[df_posts_by_time_cut.post==\
                         int(df_posts_by_time_cut['post'].max())].time_cut.min())

In [18]:
time_cut_last_post + (60*24)

2039

In [8]:
def find_best_f1_threshold(y_true, y_probs):
    """
    Find the probability threshold that yields the highest F1-score.

    Parameters
    ----------
    y_true : array-like
        Ground truth binary labels (0 or 1).
    y_probs : array-like
        Predicted probabilities for the positive class.

    Returns
    -------
    float
        Best decision threshold that maximizes F1-score.

    Notes
    -----
    - The search is done by evaluating 20 thresholds between 0.05 and 1.0.
    - F1-score is optimized for imbalanced datasets since it balances 
      precision and recall.
    """
    thresholds = np.linspace(0.05, 1, 20)
    best_thresh = 0.5
    best_f1 = 0

    for thresh in thresholds:
        preds = (y_probs >= thresh).astype(int)

        # Compute precision, recall, and F1-score for binary classification
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, preds, average="binary"
        )

        # Track best performing threshold
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    return best_thresh


def evaluate(model, loader):
    """
    Run inference on a dataset and compute AUC for model performance.

    Parameters
    ----------
    model : torch.nn.Module
        Trained binary classification model.
    loader : DataLoader
        Data loader containing input tensors and true labels.

    Returns
    -------
    labels : numpy.ndarray
        Ground truth labels collected from the dataset.
    probs : numpy.ndarray
        Predicted probabilities after applying sigmoid.
    auc : float
        ROC-AUC score representing discrimination performance.

    Notes
    -----
    - Model is set to evaluation mode and gradients are disabled.
    - The model's output logits are converted to probabilities using sigmoid.
    - AUC is threshold-independent and ideal for imbalanced datasets.
    """
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            output = model(X_batch).view(-1)  # Convert to 1D logits

            # Store raw logits and true labels
            all_logits.extend(output.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    # Convert logits to probabilities
    probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
    labels = np.array(all_labels)

    # Compute ROC-AUC
    auc = roc_auc_score(labels, probs)

    return labels, probs, auc


* **The initial  time cut will be 10 minutes after the first post publication**
*  **The final time cut will be equal to 6 hours after the publication of last post**

In [ ]:
previous_node_count = 0

for time_cut in range(10, max_time_cut+(60*6), 10):
    print(f"\n=== Time Cut: {time_cut} ===")
    
    train_dataset = 'charlie_hebdo'
    #test_dataset = 'sydneysiege'
    test_dataset = 'ottawashooting'
    #test_dataset = 'germanwings_crash'
    #test_dataset = 'ferguson'
    time_cut =time_cut
    processor = Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning(train_dataset,\
               test_dataset, time_cut=time_cut,test_size=0.7)
    
    processor.load_data()
    processor.process_data()
    train,test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test.values, dtype=torch.long)
    X_test_new = torch.tensor(X_test_new, dtype=torch.float32)
    y_test_new = torch.tensor(y_test_new.values, dtype=torch.long)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    test_new_dataset = TensorDataset(X_test_new, y_test_new)

    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader_new = DataLoader(test_new_dataset, batch_size=32)
    test_loader = DataLoader(test_dataset, batch_size=32)




    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):


        model = RumorDetectionLSTM()
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        epochs = 200
        
        for epoch in range(epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                output = model(X_batch).view(-1)

                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()

            # Evaluate at every epoch
            y_train_true, y_train_probs, auc_train = evaluate(model, train_loader)
            y_test_true, y_test_probs, auc_test = evaluate(model, test_loader)
            y_test_new_true, y_test_new_probs, auc_test_new = evaluate(model, test_loader)

            best_thresh = find_best_f1_threshold(y_train_true, y_train_probs)

        def compute_metrics(y_true, y_probs, threshold):
            preds = (y_probs >= threshold).astype(int)
            precision, recall, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary')
            return precision, recall, f1

        best_thresh = find_best_f1_threshold(y_train_true, y_train_probs)

        prec_train, rec_train, f1_train = compute_metrics(y_train_true, y_train_probs, best_thresh)
        prec_test, rec_test, f1_test = compute_metrics(y_test_true, y_test_probs, best_thresh)
        prec_test_new, rec_test_new, f1_test_new = compute_metrics(y_test_new_true, y_test_new_probs, best_thresh)
        print(f"Epoch {epoch+1}")
        print(f"  Train AUC: {auc_train:.4f} | Precision: {prec_train:.4f} | Recall: {rec_train:.4f} | F1: {f1_train:.4f}")
        print(f"  Test  AUC: {auc_test:.4f} | Precision: {prec_test:.4f} | Recall: {rec_test:.4f} | F1: {f1_test:.4f}")
        print(f"  Threshold: {best_thresh:.2f}\n")



         # Log final test results
        mlflow.log_metric("final_precision", prec_test)
        mlflow.log_metric("final_recall", rec_test)
        mlflow.log_metric("final_f1", f1_test)
        mlflow.log_metric("final_auc", auc_test)
        mlflow.log_metric("time_cut", time_cut)


        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", prec_test_new)
            mlflow.log_metric("curr_recall",rec_test_new)
            mlflow.log_metric("curr_f1",f1_test_new)
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_f1", 0)


=== Time Cut: 5335 ===
rumour
0    156
1    127
Name: count, dtype: int64
New Instances: 283
Epoch 200
  Train AUC: 1.0000 | Precision: 0.9924 | Recall: 1.0000 | F1: 0.9962
  Test  AUC: 0.7006 | Precision: 0.5765 | Recall: 0.7717 | F1: 0.6599
  Threshold: 0.55


=== Time Cut: 5350 ===
rumour
0    156
1    127
Name: count, dtype: int64
New Instances: 0
Epoch 200
  Train AUC: 1.0000 | Precision: 0.9905 | Recall: 1.0000 | F1: 0.9952
  Test  AUC: 0.6926 | Precision: 0.5706 | Recall: 0.7638 | F1: 0.6532
  Threshold: 0.55


=== Time Cut: 5365 ===
rumour
0    156
1    127
Name: count, dtype: int64
New Instances: 0
Epoch 200
  Train AUC: 1.0000 | Precision: 0.9924 | Recall: 1.0000 | F1: 0.9962
  Test  AUC: 0.6878 | Precision: 0.5749 | Recall: 0.7559 | F1: 0.6531
  Threshold: 0.55


=== Time Cut: 5380 ===
rumour
0    156
1    127
Name: count, dtype: int64
New Instances: 0
Epoch 200
  Train AUC: 1.0000 | Precision: 0.9981 | Recall: 1.0000 | F1: 0.9990
  Test  AUC: 0.6353 | Precision: 0.5059 | R